# 07 - Comparacion App vs Experto (severidad y patogeno)

Lee un CSV con las salidas ya registradas de la app y del experto, y las compara contra la verdad de campo (`patogeno_real` = carpeta). Objetivo: mostrar si la app es **igual o mas precisa que el experto** detectando la enfermedad, y la concordancia de severidad app-experto.

**Entrada:** `splits/severity_val/labels.csv` con columnas:
`file_name, severidad_experto, severidad_app, patogeno_experto, patogeno_app, patogeno_real`

- `severidad_*` en % (0-100). `patogeno_*` en {bacterianas, fungicas, plagas_insectos, roya, virales, sana}.
- `patogeno_real` = nombre de la carpeta (verdad de campo). Si falta la columna, se deduce de la subcarpeta de la imagen.
- Un unico evaluador experto (controla la variabilidad inter-evaluador).

**Salidas:** `comparacion_app_vs_experto.json`, `severidad_app_vs_experto.png`, `patogeno_app_vs_experto.png`.

In [ ]:
!pip install -q pandas numpy matplotlib scipy scikit-learn

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score

SPLIT = Path('./splits'); OUT = Path('./outputs'); OUT.mkdir(exist_ok=True)
SEV_DIR = SPLIT / 'severity_val'
CSV = SEV_DIR / 'labels.csv'

df = pd.read_csv(CSV)
df.columns = [c.strip().lower() for c in df.columns]

if 'patogeno_real' not in df.columns:
    folder = {}
    for sub in (SEV_DIR / 'images').iterdir():
        if sub.is_dir():
            for f in sub.iterdir():
                folder[f.name] = sub.name
    df['patogeno_real'] = df['file_name'].map(lambda n: folder.get(str(n)))

for col in ['patogeno_experto', 'patogeno_app', 'patogeno_real']:
    df[col] = df[col].astype(str).str.strip().str.lower()
for col in ['severidad_experto', 'severidad_app']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'n = {len(df)} hojas | clases reales: {sorted(df.patogeno_real.unique())}')
df.head()

In [ ]:
sev = df.dropna(subset=['severidad_experto', 'severidad_app'])
exp_s = sev['severidad_experto'].to_numpy(float)
app_s = sev['severidad_app'].to_numpy(float)

r = float(np.corrcoef(exp_s, app_s)[0, 1]) if len(exp_s) > 1 else float('nan')
mx, my = exp_s.mean(), app_s.mean()
vx, vy = exp_s.var(), app_s.var()
cov = ((exp_s - mx) * (app_s - my)).mean()
ccc = float(2 * cov / (vx + vy + (mx - my) ** 2)) if (vx + vy) > 0 else float('nan')
mae = float(np.mean(np.abs(app_s - exp_s)))
rmse = float(np.sqrt(np.mean((app_s - exp_s) ** 2)))
diff = app_s - exp_s
bias = float(diff.mean()); sd = float(diff.std(ddof=1)) if len(diff) > 1 else 0.0
loa_lo, loa_hi = bias - 1.96 * sd, bias + 1.96 * sd
sev_metrics = {'n': int(len(exp_s)), 'pearson_r': round(r, 3), 'ccc': round(ccc, 3),
               'mae': round(mae, 2), 'rmse': round(rmse, 2), 'bias': round(bias, 2),
               'loa_low': round(loa_lo, 2), 'loa_high': round(loa_hi, 2),
               'within_10pct': round(float(np.mean(np.abs(diff) <= 10) * 100), 1),
               'within_20pct': round(float(np.mean(np.abs(diff) <= 20) * 100), 1)}
print('Severidad app vs experto:', json.dumps(sev_metrics, ensure_ascii=False))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.scatter(exp_s, app_s, alpha=0.7, color='steelblue')
a1.plot([0, 100], [0, 100], 'k--', lw=1, label='identidad')
a1.set_xlim(0, 100); a1.set_ylim(0, 100)
a1.set_xlabel('Severidad experto (%)'); a1.set_ylabel('Severidad app (%)')
a1.set_title(f'App vs experto (r={r:.3f}, CCC={ccc:.3f}, n={len(exp_s)})')
a1.legend(); a1.grid(alpha=0.3)
mean_pair = (exp_s + app_s) / 2
a2.scatter(mean_pair, diff, alpha=0.7, color='steelblue')
a2.axhline(bias, color='red', label=f'sesgo {bias:.1f}%')
a2.axhline(loa_hi, color='gray', ls='--', label=f'LoA 95% [{loa_lo:.1f}, {loa_hi:.1f}]')
a2.axhline(loa_lo, color='gray', ls='--')
a2.set_xlabel('Media app-experto (%)'); a2.set_ylabel('App - experto (%)')
a2.set_title('Bland-Altman'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / 'severidad_app_vs_experto.png', dpi=120); plt.show()

In [ ]:
real = df['patogeno_real'].to_numpy()
app = df['patogeno_app'].to_numpy()
expert = df['patogeno_experto'].to_numpy()

acc_app = float(np.mean(app == real))
acc_exp = float(np.mean(expert == real))
classes = sorted(set(real))
f1_app = float(f1_score(real, app, labels=classes, average='macro', zero_division=0))
f1_exp = float(f1_score(real, expert, labels=classes, average='macro', zero_division=0))

app_ok = app == real; exp_ok = expert == real
b = int(np.sum(exp_ok & ~app_ok))
c = int(np.sum(app_ok & ~exp_ok))
try:
    from scipy.stats import binomtest
    mcnemar_p = float(binomtest(min(b, c), b + c, 0.5).pvalue) if (b + c) > 0 else 1.0
except Exception:
    mcnemar_p = 1.0

pat_metrics = {'n': int(len(real)), 'accuracy_app': round(acc_app, 3), 'accuracy_experto': round(acc_exp, 3),
               'f1_macro_app': round(f1_app, 3), 'f1_macro_experto': round(f1_exp, 3),
               'discordantes_solo_experto': b, 'discordantes_solo_app': c, 'mcnemar_p': round(mcnemar_p, 4)}
print('Patogeno:', json.dumps(pat_metrics, ensure_ascii=False))
veredicto = 'APP >= EXPERTO' if acc_app >= acc_exp else 'EXPERTO > APP'
sig = 'significativa (p<0.05)' if mcnemar_p < 0.05 else 'no significativa'
print(f'  {veredicto} | diferencia {sig}')

json.dump({'severidad': sev_metrics, 'patogeno': pat_metrics},
          open(OUT / 'comparacion_app_vs_experto.json', 'w'), indent=2, ensure_ascii=False)

In [ ]:
acc_app_c, acc_exp_c = [], []
for cl in classes:
    m = real == cl
    acc_app_c.append(float(np.mean(app[m] == real[m])))
    acc_exp_c.append(float(np.mean(expert[m] == real[m])))
labels_plot = classes + ['GLOBAL']
acc_app_c.append(acc_app); acc_exp_c.append(acc_exp)

x = np.arange(len(labels_plot)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w / 2, acc_app_c, w, label=f'App (acc {acc_app:.2f})', color='seagreen')
b2 = ax.bar(x + w / 2, acc_exp_c, w, label=f'Experto (acc {acc_exp:.2f})', color='slategray')
ax.set_xticks(x); ax.set_xticklabels(labels_plot, rotation=20)
ax.set_ylim(0, 1.05); ax.set_ylabel('Exactitud vs patogeno real')
ax.set_title(f'Patogeno: App vs Experto vs Real (n={len(real)}, McNemar p={pat_metrics["mcnemar_p"]})')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bars in (b1, b2):
    for rect in bars:
        ax.annotate(f'{rect.get_height():.2f}', (rect.get_x() + rect.get_width() / 2, rect.get_height()),
                    ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(OUT / 'patogeno_app_vs_experto.png', dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, title in [(axes[0], app, 'App'), (axes[1], expert, 'Experto')]:
    cm = confusion_matrix(real, pred, labels=classes)
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel('predicho'); ax.set_ylabel('real'); ax.set_title(f'{title} vs real')
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.tight_layout(); plt.savefig(OUT / 'patogeno_confusion_app_experto.png', dpi=120); plt.show()